# Neutral-atom resource study: does routing-awareness pay off?

**Pack:** `samples/research/` (#190) — literate notebook, paired with
[`na_resource_study_smoke.py`](./na_resource_study_smoke.py).

**Question:** the zoned NA placer supports two modes,
`routing-agnostic` (default) and `routing-aware`. Does routing-awareness
reduce the analytic resource report's `estimated_cycles` for a real
multi-qubit workload — and is "no placement freedom" (a 2-qubit Bell pair)
really placer-invariant?

**Linked canonical sources & artifacts:**
[`test/na/bell.qn`](../../test/na/bell.qn),
[`test/na/qaoa_graph.qn`](../../test/na/qaoa_graph.qn) — both already carry
`ci: smoke` catalog rows (`neutral-atom/bell-pair`, `neutral-atom/qaoa-maxcut`,
#192). Their `routing-agnostic` resource reports are also already checked in
as #189 goldens at
[`samples/visualization/goldens/na_schedule_metrics/`](../visualization/goldens/na_schedule_metrics/);
this notebook cross-checks against those rather than re-deriving them, and
only compiles *new* cells (the `routing-aware` mode, not present in any
checked-in golden).

## Setup

```bash
cargo build --release -p quonc
export QUONC=$PWD/target/release/quonc
```

Every `--emit-resource-report` invocation below is analytic
(`evidence_kind: analytic`, ADR-0020) — not a sampled/Sinter claim, and not a
threshold claim. As with every notebook in this pack, code cells show real
commands with no fabricated outputs — see
[`repro_appendix_template.md`](./repro_appendix_template.md). The numbers in
the prose come from running
[`na_resource_study_smoke.py`](./na_resource_study_smoke.py) against commit
`d26723141d5799a0e8107915b16c55ef3b9fa6f3`.

## Cell 1: `bell.qn` — no placement freedom

Two qubits, one `CNOT`. There is only one way to place two atoms that need
to interact once, so both placer modes should produce byte-identical
analytic reports.

In [ ]:
$QUONC --target targets/neutral_atom/generic_rna_v0.json \
       --na-backend zoned --na-placer routing-agnostic --verify-na \
       --emit-resource-report - -q test/na/bell.qn
# estimated_cycles=9 rearrangement_steps=1 trap_transfers=4 bottleneck=rearrangement

In [ ]:
$QUONC --target targets/neutral_atom/generic_rna_v0.json \
       --na-backend zoned --na-placer routing-aware --verify-na \
       --emit-resource-report - -q test/na/bell.qn
# identical: estimated_cycles=9 rearrangement_steps=1 trap_transfers=4 bottleneck=rearrangement

**Result:** confirmed — every analytic field is identical across placer
modes for `bell.qn`. This is the sanity/degenerate-case check: with no
placement choice to make, "routing-aware" has nothing to be aware of.

## Cell 2: `qaoa_graph.qn` — a 4-qubit dense interaction graph

`qaoa_graph.qn` is a 3-regular MaxCut-style layer ($\Delta \approx 3$; see
its header comment) — already low-degree, so a routing-aware placer's
locality-seeking heuristic has less headroom to exploit than on a denser or
higher-degree graph.

In [ ]:
$QUONC --target targets/neutral_atom/generic_rna_v0.json \
       --na-backend zoned --na-placer routing-agnostic --verify-na \
       --emit-resource-report - -q test/na/qaoa_graph.qn
# estimated_cycles=80 rearrangement_steps=8 trap_transfers=22 bottleneck=rearrangement

In [ ]:
$QUONC --target targets/neutral_atom/generic_rna_v0.json \
       --na-backend zoned --na-placer routing-aware --verify-na \
       --emit-resource-report - -q test/na/qaoa_graph.qn
# estimated_cycles=83 rearrangement_steps=9 trap_transfers=24 bottleneck=rearrangement

**Result — counterintuitive:** `routing-aware` costs *more* than
`routing-agnostic` on this fixture: 83 vs. 80 estimated cycles, 9 vs. 8
rearrangement steps, 24 vs. 22 trap transfers. The bottleneck stays
`rearrangement` either way.

### Resource table

| Fixture | Placer | `estimated_cycles` | `rearrangement_steps` | `trap_transfers` |
| --- | --- | --- | --- | --- |
| `bell.qn` | routing-agnostic | 9 | 1 | 4 |
| `bell.qn` | routing-aware | 9 | 1 | 4 |
| `qaoa_graph.qn` | routing-agnostic | 80 | 8 | 22 |
| `qaoa_graph.qn` | routing-aware | 83 | 9 | 24 |

### Why routing-awareness doesn't pay off here

Routing-aware placement optimizes for a different objective than raw
rearrangement-step count — it tries to keep frequently-interacting logical
qubits physically close *throughout* the circuit, which can cost extra
transfers up front on a graph that's already low-degree enough for the
row-major default to do reasonably well. This is *not* evidence that
routing-aware placement is broken or never helps: it's evidence that its
benefit is workload-dependent, and this particular fixture ($\Delta \approx
3$, uniform edge weights) doesn't have the structure (e.g. hub qubits with
much higher degree than their neighbors) that would let locality-aware
placement pay for its own bookkeeping. A follow-up notebook could re-run
this same comparison against a higher-degree or non-uniform interaction
graph to test that hypothesis — left as a future extension, not claimed
here.

## Repro appendix

- **Quon commit:** `d26723141d5799a0e8107915b16c55ef3b9fa6f3` (branch
  `samples/190-research-notebooks`)
- **`quonc` version:** `quonc 0.1.0`
- **Build:** `cargo build --release -p quonc`
- **Python:** none required — this notebook's smoke twin is pure `stdlib`
  (`json`, `subprocess`) plus `quonc` itself
- **Smoke twin:** `python samples/research/na_resource_study_smoke.py`
- **Linked canonical sources & artifacts:** `test/na/bell.qn`
  (`neutral-atom/bell-pair`), `test/na/qaoa_graph.qn`
  (`neutral-atom/qaoa-maxcut`);
  `samples/visualization/goldens/na_schedule_metrics/{bell_zoned,qaoa_graph_zoned}.resource_report.json`
  (#189, cross-checked, not forked)